In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import StandardScaler
from sklearn import cluster
from sklearn.cluster import KMeans
from google.colab import drive

drive.mount('/content/drive')
# 4. Incluye las librerías y guardar el archivo en una variable llamada "data"
data = pd.read_csv('/content/drive/MyDrive/2025/Aprendizaje supervisado/Reto/bank_marketing.csv')

Mounted at /content/drive


In [ ]:
data.head(4)

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,31,self-employed,married,tertiary,no,2666,no,no,cellular,10,nov,318,2,97,6,success,yes
1,29,unemployed,single,unknown,no,1584,no,no,cellular,6,sep,245,1,-1,0,unknown,yes
2,41,blue-collar,married,secondary,no,2152,yes,no,cellular,17,nov,369,1,-1,0,unknown,no
3,50,blue-collar,married,secondary,no,84,yes,no,cellular,17,jul,18,8,-1,0,unknown,no


In [ ]:
#Obtener la información de dicha base de datos que incluya numero de registros, el total de variables,
#el tipo de cada variable, la cantidad de datos perdidos de cada variables en caso de exista
# Se tiene 17 columnas y 9,000 datos los tipos de datos son entero y objeto, de los cuales
#inferimos que
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        9000 non-null   int64 
 1   job        9000 non-null   object
 2   marital    9000 non-null   object
 3   education  9000 non-null   object
 4   default    9000 non-null   object
 5   balance    9000 non-null   int64 
 6   housing    9000 non-null   object
 7   loan       9000 non-null   object
 8   contact    9000 non-null   object
 9   day        9000 non-null   int64 
 10  month      9000 non-null   object
 11  duration   9000 non-null   int64 
 12  campaign   9000 non-null   int64 
 13  pdays      9000 non-null   int64 
 14  previous   9000 non-null   int64 
 15  poutcome   9000 non-null   object
 16  y          9000 non-null   object
dtypes: int64(7), object(10)
memory usage: 1.2+ MB


In [ ]:
data.nunique()
#Las variables categoricas son job, marital, education, default,
# housing, loan, contact, month, poutcome

,0
age,74
job,12
marital,3
education,4
default,2
balance,3476
housing,2
loan,2
contact,3
day,31


In [ ]:
X = data[['age','job', 'marital', 'education', 'default', 'balance','housing', 'loan', 'contact', 'day','month', 'duration', 'campaign','pdays', 'previous' , 'poutcome']]
Y = data['y']

In [ ]:
X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9000 entries, 0 to 8999
Data columns (total 16 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        9000 non-null   int64 
 1   job        9000 non-null   object
 2   marital    9000 non-null   object
 3   education  9000 non-null   object
 4   default    9000 non-null   object
 5   balance    9000 non-null   int64 
 6   housing    9000 non-null   object
 7   loan       9000 non-null   object
 8   contact    9000 non-null   object
 9   day        9000 non-null   int64 
 10  month      9000 non-null   object
 11  duration   9000 non-null   int64 
 12  campaign   9000 non-null   int64 
 13  pdays      9000 non-null   int64 
 14  previous   9000 non-null   int64 
 15  poutcome   9000 non-null   object
dtypes: int64(7), object(9)
memory usage: 1.1+ MB


In [ ]:
Y.info()

<class 'pandas.core.series.Series'>
RangeIndex: 9000 entries, 0 to 8999
Series name: y
Non-Null Count  Dtype 
--------------  ----- 
9000 non-null   object
dtypes: object(1)
memory usage: 70.4+ KB


In [ ]:
X['job'].value_counts()

,count
job,
management,1999
blue-collar,1688
technician,1485
admin.,1004
services,771
retired,594
self-employed,319
student,312
unemployed,290


In [ ]:
X['contact'].value_counts()

,count
contact,
cellular,6438
unknown,1982
telephone,580


In [ ]:
# Transformación de variables categóricas de manera que cada nivel quede registrado con un entero
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
cols = ['job', 'marital', 'education', 'default','housing','loan','contact','month','poutcome']
X[cols] = X[cols].apply(LabelEncoder().fit_transform)
X.head()

/tmp/ipython-input-1937095693.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[cols] = X[cols].apply(LabelEncoder().fit_transform)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome
0,31,6,1,2,0,2666,0,0,0,10,9,318,2,97,6,2
1,29,10,2,3,0,1584,0,0,0,6,11,245,1,-1,0,3
2,41,1,1,1,0,2152,1,0,0,17,9,369,1,-1,0,3
3,50,1,1,1,0,84,1,0,0,17,5,18,8,-1,0,3
4,40,0,1,1,0,0,0,0,0,28,5,496,2,182,11,2


In [ ]:
Y.value_counts()
#De donde se determina que hay 3,787 creditos aprobados y 5,213 que no fueron aprobados.

,count
y,
no,5213
yes,3787


In [ ]:
#9. Particiona los datos en los conjuntos de entrenamiento, validación y prueba en 60%, 20% y 20%, respectivamente.
from sklearn.model_selection import train_test_split

x_train_validation, x_test, y_train_validation, y_test = \
train_test_split(X, Y, test_size=.20)

x_train, x_validation, y_train, y_validation = \
train_test_split(x_train_validation, y_train_validation, test_size=0.250)

In [ ]:
print("Dimensión X_train_set:", x_train.shape)
print("Dimensión X_validation_set:\t", x_validation.shape)
print("Dimensión X_test_set:\t\t", x_test.shape)
print("\nDimensión Y_train:\t\t", y_train.shape)
print("Dimensión Y_validation:\t\t", y_validation.shape)
print("Dimensión Y_test:\t\t", y_test.shape)

Dimensión X_train_set: (5400, 16)
Dimensión X_validation_set:	 (1800, 16)
Dimensión X_test_set:		 (1800, 16)

Dimensión Y_train:		 (5400,)
Dimensión Y_validation:		 (1800,)
Dimensión Y_test:		 (1800,)


In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression( C=1.0, solver='newton-cg')

modelo_RL = clf.fit( x_train, y_train )

print("Regresión Logística:\nExactitud (accuracy) con el conjunto de Validación = ", modelo_RL.score(x_validation, y_validation))

#Se obtuvo 79% de exactitud en tu primera aproximación del modelo de regresión logística
#con el conjunto de validación.

Regresión Logística:
Exactitud (accuracy) con el conjunto de Validación =  0.7922222222222223


In [ ]:
#Matriz de confusión
from sklearn.metrics import confusion_matrix
pr = modelo_RL.predict(x_validation)
confusion_matrix(y_validation, pr)

array([[175, 861],
       [ 56, 708]])

In [ ]:
#11. Aplica el modelo Red Neuronal en el conjunto de entrenamiento. Valida el modelo con las predicciones del conjunto de validación y su matriz de confusión.
#Ajusta los parámetros del modelo hasta obtener tu mejor modelo, entre ellos el número de neuronas y capas  ocultas.
from sklearn.neural_network import MLPClassifier
modelo_NN = MLPClassifier(hidden_layer_sizes=(15, 4), max_iter=3787)
modelo_NN.fit(x_train, y_train)

MLPClassifier(hidden_layer_sizes=(15, 4), max_iter=3787)

In [ ]:
print(modelo_NN.score(x_validation, y_validation))

0.76


In [ ]:
#Matriz de confusion
print(confusion_matrix(y_test, pr))

[[645 390]
 [473 292]]
